# Hybrid GraphSAGE + CNN Fraud Detector (GraphSAGE + 1D-CNN) — runnable, leakage-aware version

What this notebook does differently from the first version of the repo:

| Problem in v1 | Fix here |
|---|---|
| Random node features / random targets | Real features from amounts, balances, types, degrees |
| No training | Training loop with **focal loss**, early stopping on validation PR-AUC |
| Random "ground truth" in evaluation | Real `isFraud` labels, **time-based** train/val/test split |
| First 50k rows (almost no fraud) | Stratified sample that **keeps every fraud row** |
| Bipartite SAGEConv index bug | One **homogeneous** account graph (shared node index) |
| Account-level scores | **Transaction-level** scores (the paper also scores transactions) |
| Random temporal tensor | Real last-K sequence per originating account |
| No baseline | Gradient-boosting baseline on tabular features |
| ROC-AUC only | **PR-AUC**, ROC-AUC, precision/recall/F1 at threshold, threshold sweep |
| Neo4j: wrong user, `MERGE` collapses edges, one giant UNWIND | Optional, batched `CREATE`, credentials from env/`getpass` |

**Data:** put PaySim's CSV at `data/01_raw/Fraud Detection Dataset.csv` (or `PS_20174392719_1491204439457_log.csv`). If no file is found, the notebook generates a PaySim-*style* synthetic dataset with injected fraud rings so you can verify the pipeline end to end. Synthetic results are a smoke test, **not** evidence of real-world performance.

In [ ]:
# --- Install (Colab/local). Safe to re-run. Restart runtime only if Colab asks. ---
import importlib, subprocess, sys
def _need(mod): 
    try: importlib.import_module(mod); return False
    except ImportError: return True
pkgs = {"torch_geometric":"torch-geometric","pyvis":"pyvis","networkx":"networkx","sklearn":"scikit-learn","neo4j":"neo4j"}
missing = [p for m,p in pkgs.items() if _need(m)]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
print("deps ok", "(installed: %s)"%missing if missing else "")

In [ ]:
import os, glob, random, time, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             precision_score, recall_score, f1_score)
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")

CFG = dict(
    seed=42,
    raw_dir="data/01_raw",
    max_rows=200_000,          # RAM guard: keep ALL fraud rows + sample of legit rows up to this size
    seq_len=10,                # last-K transactions per originating account
    hidden=64, gnn_out=32, cnn_ch=32, dropout=0.3, lr=3e-3, weight_decay=1e-4,
    epochs=60, patience=10,
    focal_alpha=0.75, focal_gamma=2.0,
    decision_threshold=None,   # None = pick on validation set (max F1). Or set e.g. 0.75 for a fixed risk appetite
    out_dir="data/07_model_output",
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(CFG["out_dir"], exist_ok=True)
print("device:", DEVICE)

## 1. Load data (real PaySim if present, otherwise synthetic smoke-test data)

In [ ]:
def make_synthetic_paysim(n_legit=60_000, n_accounts=20_000, n_rings=12, ring_size=6, seed=0):
    """PaySim-style data with injected laundering rings (cycles of TRANSFER -> CASH_OUT) + a few mule fan-ins.
    ONLY for verifying the pipeline. Fraud here is easy to find by construction."""
    rng = np.random.default_rng(seed)
    accts = np.array([f"C{i:09d}" for i in range(n_accounts)])
    merch = np.array([f"M{i:07d}" for i in range(3000)])
    rows = []
    # legit traffic
    for _ in range(n_legit):
        step = int(rng.integers(1, 700)); t = rng.choice(["PAYMENT","CASH_OUT","CASH_IN","TRANSFER","DEBIT"], p=[.5,.2,.15,.1,.05])
        o = rng.choice(accts); d = rng.choice(merch) if t=="PAYMENT" else rng.choice(accts)
        amt = float(rng.lognormal(8.0, 1.0)); ob = float(rng.lognormal(10, 1.2)); 
        rows.append((step,t,amt,o,ob,max(ob-amt,0) if t!="CASH_IN" else ob+amt,d,float(rng.lognormal(10,1.2)),0.0,0))
    # fraud rings: money hops around a cycle with near-total balance drain
    for r in range(n_rings):
        members = rng.choice(accts, ring_size, replace=False)
        start = int(rng.integers(50, 650))
        for rep in range(3):
            for i in range(ring_size):
                o, d = members[i], members[(i+1)%ring_size]
                amt = float(rng.lognormal(11.5, .4)); ob = amt*float(rng.uniform(1.0,1.05))
                rows.append((start+rep*4+i, "TRANSFER" if i%2==0 else "CASH_OUT", amt, o, ob, ob-amt, d, 0.0, amt if i%2==0 else 0.0, 1))
    df = pd.DataFrame(rows, columns=["step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig",
                                     "nameDest","oldbalanceDest","newbalanceDest","isFraud"])
    df["isFlaggedFraud"] = 0
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

def load_raw():
    for pat in ["Fraud Detection Dataset.csv", "PS_*.csv", "*.csv"]:
        hits = sorted(glob.glob(os.path.join(CFG["raw_dir"], pat)))
        hits = [h for h in hits if "synthetic_sample" not in h] or hits
        if hits:
            print("Loading real data:", hits[0]); return pd.read_csv(hits[0]), True
    print("No CSV in", CFG["raw_dir"], "-> generating SYNTHETIC smoke-test data"); return make_synthetic_paysim(), False

raw, IS_REAL = load_raw()
need = ["step","type","amount","nameOrig","oldbalanceOrg","newbalanceOrig","nameDest","oldbalanceDest","newbalanceDest","isFraud"]
assert set(need) <= set(raw.columns), f"Missing columns: {set(need)-set(raw.columns)}"
print(raw.shape, "| fraud rate: %.4f%%" % (100*raw.isFraud.mean()), "| fraud rows:", int(raw.isFraud.sum()))

In [ ]:
# Stratified sampling: keep ALL fraud rows, subsample legit rows. Then sort by time.
# (Sampling changes the base rate, so report PR-AUC with that in mind; see 'Prevalence' in the results.)
fraud = raw[raw.isFraud==1]; legit = raw[raw.isFraud==0]
n_legit_keep = max(min(len(legit), CFG["max_rows"]-len(fraud)), 0)
df = pd.concat([fraud, legit.sample(n=n_legit_keep, random_state=CFG["seed"])])
df = df.sort_values("step", kind="stable").reset_index(drop=True)
print("working set:", df.shape, "| fraud rate: %.3f%%" % (100*df.isFraud.mean()))

# Time-based split on `step` (60/20/20 by transaction count) — never random.
q = df.step.quantile([.6,.8]).values
df["split"] = np.where(df.step<=q[0],"train",np.where(df.step<=q[1],"val","test"))
print(df.groupby("split").agg(n=("isFraud","size"), fraud=("isFraud","sum"), step_min=("step","min"), step_max=("step","max")))
for s in ["train","val","test"]:
    assert df.loc[df.split==s,"isFraud"].sum()>=5, f"Too few fraud rows in {s}: increase max_rows or use more data"

## 2. Feature engineering

- **Transaction features:** log amount, balance deltas, *balance-error* features (`oldbal - amount - newbal`), type one-hot, hour-of-day.
- **Node features** (built per snapshot using only transactions up to a cutoff step, so no future information leaks into training): in/out degree, amount sums/means, unique counterparties, fraction of sent money that drains the balance.
- **Sequence tensor:** the last `K` prior transactions of the *same originator* → `[E, channels, K]` for the 1D-CNN.

> ⚠️ PaySim caveat: the balance-error features are extremely predictive in PaySim (they are a simulator artifact). That's why the tabular baseline will look very strong. Report this honestly; the interesting question is whether graph context adds anything *beyond* it.

In [ ]:
TYPES = ["CASH_IN","CASH_OUT","DEBIT","PAYMENT","TRANSFER"]

# unified account index (senders AND receivers share one node space -> fixes the bipartite index bug)
nodes = pd.Index(pd.unique(np.concatenate([df.nameOrig.values, df.nameDest.values])))
df["src"] = nodes.get_indexer(df.nameOrig); df["dst"] = nodes.get_indexer(df.nameDest)
N = len(nodes); E = len(df); print("nodes:", N, "edges:", E)

def tx_features(d):
    f = pd.DataFrame(index=d.index)
    f["log_amt"]       = np.log1p(d.amount)
    f["orig_delta"]    = np.sign(d.oldbalanceOrg-d.newbalanceOrig)*np.log1p((d.oldbalanceOrg-d.newbalanceOrig).abs())
    f["dest_delta"]    = np.sign(d.newbalanceDest-d.oldbalanceDest)*np.log1p((d.newbalanceDest-d.oldbalanceDest).abs())
    f["err_orig"]      = np.sign(d.oldbalanceOrg-d.amount-d.newbalanceOrig)*np.log1p((d.oldbalanceOrg-d.amount-d.newbalanceOrig).abs())
    f["err_dest"]      = np.sign(d.oldbalanceDest+d.amount-d.newbalanceDest)*np.log1p((d.oldbalanceDest+d.amount-d.newbalanceDest).abs())
    f["drain_ratio"]   = (d.amount/(d.oldbalanceOrg+1)).clip(0,2)
    f["orig_zeroed"]   = ((d.newbalanceOrig==0)&(d.oldbalanceOrg>0)).astype(float)
    f["hour"]          = (d.step%24)/23.0
    for t in TYPES: f["t_"+t] = (d.type==t).astype(float)
    return f
TXF = tx_features(df)
FEATS = list(TXF.columns)

# standardise using TRAIN statistics only
tr_mask = (df.split=="train").values
mu, sd = TXF[tr_mask].mean(), TXF[tr_mask].std().replace(0,1)
TX = ((TXF-mu)/sd).values.astype(np.float32)
print("tx feature dim:", TX.shape[1])

In [ ]:
def node_features(max_step):
    """Per-account stats from transactions with step <= max_step only (no future leakage)."""
    d = df[df.step<=max_step]
    d = d.assign(orig_zero=((d.newbalanceOrig==0)&(d.oldbalanceOrg>0)).astype(float))
    out_g = d.groupby("src").agg(out_deg=("amount","size"), out_sum=("amount","sum"), out_mean=("amount","mean"),
                                 out_uniq=("dst","nunique"), out_drain=("orig_zero","mean"))
    in_g  = d.groupby("dst").agg(in_deg=("amount","size"), in_sum=("amount","sum"), in_mean=("amount","mean"),
                                 in_uniq=("src","nunique"))
    X = pd.DataFrame(index=np.arange(N)).join(out_g).join(in_g).fillna(0.0)
    for c in ["out_deg","out_sum","out_mean","out_uniq","in_deg","in_sum","in_mean","in_uniq"]: X[c]=np.log1p(X[c])
    return X.values.astype(np.float32), d[["src","dst"]].values.T

steps = {s: int(df.loc[df.split==s,"step"].max()) for s in ["train","val","test"]}
# snapshots: structure/features visible at each stage's end
SNAP = {}
for s in ["train","val","test"]:
    X, ei = node_features(steps[s]); SNAP[s] = (X, torch.tensor(ei, dtype=torch.long))
Xtr = SNAP["train"][0]; nmu, nsd = Xtr.mean(0), Xtr.std(0)+1e-6   # standardise with train snapshot stats
for s in SNAP: SNAP[s] = (torch.tensor((SNAP[s][0]-nmu)/nsd, dtype=torch.float32), SNAP[s][1])
print({s:(tuple(v[0].shape), tuple(v[1].shape)) for s,v in SNAP.items()})

In [ ]:
# Sequence tensor: for every transaction, the K previous transactions by the SAME originator (zero padded).
K = CFG["seq_len"]; order = np.lexsort((df.index.values, df.step.values, df.src.values))   # sort by account, time
src_s = df.src.values[order]; steps_s = df.step.values[order]
SEQ_FEATS = [FEATS.index(c) for c in ["log_amt","orig_delta","err_orig","drain_ratio","orig_zeroed"]] + [FEATS.index("t_"+t) for t in TYPES]
base = TX[order][:, SEQ_FEATS]                                   # [E, C0]
gap = np.zeros(E, dtype=np.float32)                              # time since previous tx of same account
same_prev = np.r_[False, src_s[1:]==src_s[:-1]]
gap[same_prev] = np.log1p(steps_s[1:][same_prev[1:]] - steps_s[:-1][same_prev[1:]])
base = np.concatenate([base, gap[:,None]], 1); C0 = base.shape[1]
seq_sorted = np.zeros((E, C0, K), dtype=np.float32)
pos = np.arange(E)
for k in range(1, K+1):
    ok = (pos-k>=0) & (src_s[np.clip(pos-k,0,None)]==src_s)
    seq_sorted[ok,:,k-1] = base[pos[ok]-k]
SEQ = np.zeros_like(seq_sorted); SEQ[order] = seq_sorted           # back to original row order
print("sequence tensor:", SEQ.shape, "| rows with >=1 prior tx: %.1f%%" % (100*(np.abs(SEQ).sum((1,2))>0).mean()))

## 3. Baseline: gradient boosting on tabular features (must be beaten, or reported honestly)

In [ ]:
y = df.isFraud.values.astype(np.float32)
idx = {s: np.where(df.split.values==s)[0] for s in ["train","val","test"]}

def report(name, p, ytrue, thr):
    pred = (p>=thr).astype(int)
    return dict(model=name, pr_auc=average_precision_score(ytrue,p), roc_auc=roc_auc_score(ytrue,p),
                precision=precision_score(ytrue,pred,zero_division=0), recall=recall_score(ytrue,pred,zero_division=0),
                f1=f1_score(ytrue,pred,zero_division=0), threshold=float(thr), prevalence=float(ytrue.mean()))

def best_f1_threshold(ytrue, p):
    pr, rc, th = precision_recall_curve(ytrue, p); f1 = 2*pr*rc/(pr+rc+1e-12)
    return float(th[np.nanargmax(f1[:-1])]) if len(th) else 0.5

gb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, class_weight="balanced", random_state=CFG["seed"])
gb.fit(TX[idx["train"]], y[idx["train"]])
p_val_gb, p_te_gb = gb.predict_proba(TX[idx["val"]])[:,1], gb.predict_proba(TX[idx["test"]])[:,1]
thr_gb = best_f1_threshold(y[idx["val"]], p_val_gb)
res_gb = report("GradBoost (tabular)", p_te_gb, y[idx["test"]], thr_gb)
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in res_gb.items()})

## 4. Hybrid model: GraphSAGE (structure) ⊕ 1D-CNN (account history) ⊕ transaction features → fraud score per transaction

In [ ]:
class HybridGraphCNN(nn.Module):
    def __init__(self, node_dim, tx_dim, seq_ch, seq_len, hidden, gnn_out, cnn_ch, dropout):
        super().__init__()
        self.sage1 = SAGEConv(node_dim, hidden); self.sage2 = SAGEConv(hidden, gnn_out)
        self.conv = nn.Conv1d(seq_ch, cnn_ch, kernel_size=3, padding=1); self.bn = nn.BatchNorm1d(cnn_ch)
        self.pool = nn.AdaptiveMaxPool1d(1)
        fused = 2*gnn_out + cnn_ch + tx_dim
        self.head = nn.Sequential(nn.Linear(fused,64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64,1))
        self.p = dropout
    def encode_nodes(self, x, ei):
        h = F.relu(self.sage1(x, ei)); h = F.dropout(h, self.p, self.training)
        return self.sage2(h, ei)                                   # [N, gnn_out]
    def forward(self, x, ei, src, dst, tx, seq):
        h = self.encode_nodes(x, ei)
        c = self.pool(F.relu(self.bn(self.conv(seq)))).squeeze(-1)  # [B, cnn_ch]
        z = torch.cat([h[src], h[dst], c, tx], 1)                   # concatenate sender, receiver, history and tx features
        return self.head(z).squeeze(-1)                             # logits

def focal_loss(logits, target, alpha, gamma):
    bce = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    p = torch.sigmoid(logits); pt = p*target + (1-p)*(1-target)
    a = alpha*target + (1-alpha)*(1-target)
    return (a*(1-pt)**gamma*bce).mean()

T = lambda a: torch.tensor(a).to(DEVICE)
TX_t, SEQ_t, Y_t = T(TX), T(SEQ), T(y)
SRC_t, DST_t = T(df.src.values), T(df.dst.values)
SN = {s:(SNAP[s][0].to(DEVICE), SNAP[s][1].to(DEVICE)) for s in SNAP}
IDX = {s:T(idx[s]) for s in idx}

model = HybridGraphCNN(SN["train"][0].shape[1], TX.shape[1], SEQ.shape[1], K, CFG["hidden"], CFG["gnn_out"], CFG["cnn_ch"], CFG["dropout"]).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])

def run(split, snap):
    i = IDX[split]; x, ei = SN[snap]
    return model(x, ei, SRC_t[i], DST_t[i], TX_t[i], SEQ_t[i])

best, bad, hist, best_state = -1, 0, [], None
t0 = time.time()
for ep in range(1, CFG["epochs"]+1):
    model.train(); opt.zero_grad()
    loss = focal_loss(run("train","train"), Y_t[IDX["train"]], CFG["focal_alpha"], CFG["focal_gamma"])
    loss.backward(); opt.step()
    model.eval()
    with torch.no_grad(): pv = torch.sigmoid(run("val","val")).cpu().numpy()
    val_ap = average_precision_score(y[idx["val"]], pv)
    hist.append((ep, loss.item(), val_ap))
    if val_ap > best: best, bad, best_state = val_ap, 0, {k:v.clone() for k,v in model.state_dict().items()}
    else: bad += 1
    if ep%5==0 or ep==1: print(f"epoch {ep:3d} | loss {loss.item():.4f} | val PR-AUC {val_ap:.4f}")
    if bad >= CFG["patience"]: print(f"early stop @ {ep}"); break
model.load_state_dict(best_state); print(f"best val PR-AUC {best:.4f} | {time.time()-t0:.1f}s")

In [ ]:
model.eval()
with torch.no_grad():
    p_val = torch.sigmoid(run("val","val")).cpu().numpy()
    p_te  = torch.sigmoid(run("test","test")).cpu().numpy()
thr = CFG["decision_threshold"] if CFG["decision_threshold"] is not None else best_f1_threshold(y[idx["val"]], p_val)
res_h = report("Hybrid GraphSAGE+CNN", p_te, y[idx["test"]], thr)
results = pd.DataFrame([res_gb, res_h]).set_index("model").round(4)
print("TEST SET (time-held-out)  | data:", "REAL PaySim" if IS_REAL else "SYNTHETIC smoke-test")
display(results) if "display" in globals() else print(results)
results.to_csv(f"{CFG['out_dir']}/metrics.csv")

In [ ]:
# Threshold sweep = the "dynamic risk appetite" dial for compliance teams (alerts vs. missed fraud).
yt = y[idx["test"]]; rows=[]
for t in [0.1,0.25,0.5,0.75,0.9,0.95]:
    pred = (p_te>=t).astype(int)
    rows.append(dict(threshold=t, alerts=int(pred.sum()), true_fraud_caught=int(((pred==1)&(yt==1)).sum()),
                     total_fraud=int(yt.sum()), precision=round(precision_score(yt,pred,zero_division=0),3),
                     recall=round(recall_score(yt,pred,zero_division=0),3)))
sweep = pd.DataFrame(rows); print(sweep.to_string(index=False)); sweep.to_csv(f"{CFG['out_dir']}/threshold_sweep.csv", index=False)

## 5. Topology analysis: do flagged transactions form rings? (Louvain on the *flagged* subgraph)

In [ ]:
import networkx as nx
from networkx.algorithms.community import louvain_communities

te = df.iloc[idx["test"]].copy(); te["score"] = p_te; te["flag"] = (p_te>=thr).astype(int)
fl = te[te.flag==1]
G = nx.Graph()
for s,d,sc in zip(fl.src, fl.dst, fl.score): G.add_edge(int(s), int(d), weight=float(sc))
print(f"flagged tx: {len(fl)} | nodes: {G.number_of_nodes()} | edges: {G.number_of_edges()}")

# degree sanity check (PaySim: most counterparties appear once -> few real rings; be honest about this)
deg_all = pd.concat([df.src, df.dst]).value_counts()
print("accounts with degree 1: %.1f%% | max degree: %d" % (100*(deg_all==1).mean(), deg_all.max()))

comms = louvain_communities(G, weight="weight", seed=CFG["seed"]) if G.number_of_edges() else []
comm_of = {n:i for i,c in enumerate(comms) for n in c}
# purity: how many true-fraud tx sit inside each syndicate
fl = fl.assign(comm=[comm_of.get(int(s),-1) for s in fl.src])
summary = (fl.groupby("comm").agg(n_tx=("score","size"), n_accounts=("src","nunique"), mean_score=("score","mean"),
                                  true_fraud=("isFraud","sum"), total_amount=("amount","sum"))
             .assign(precision=lambda d: d.true_fraud/d.n_tx).sort_values("total_amount", ascending=False))
print("\nTop syndicates by value:"); print(summary.head(10).round(3).to_string())
cyc = [c for c in nx.cycle_basis(G)]; print("\nindependent cycles in flagged subgraph:", len(cyc), "(a count, not a recall metric)")
summary.to_csv(f"{CFG['out_dir']}/syndicates.csv")

In [ ]:
# Interactive PyVis dashboard of flagged syndicates (opens from data/07_model_output/).
from pyvis.network import Network
net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=False, cdn_resources="in_line")
for n,i in comm_of.items():
    net.add_node(int(n), label=str(nodes[n])[-6:], group=int(i), title=f"{nodes[n]} | syndicate {i}")
for s,d,a,sc in zip(fl.src, fl.dst, fl.amount, fl.score):
    net.add_edge(int(s), int(d), value=float(sc), title=f"amount {a:,.0f} | score {sc:.2f}")
out_html = f"{CFG['out_dir']}/fraud_rings_dashboard.html"; net.write_html(out_html); print("wrote", out_html)

## 6. Optional: persist the graph to Neo4j AuraDB

Skipped automatically unless credentials exist. **Never paste the password into a cell** — use environment variables / Colab Secrets / `getpass`.
Newer Aura instances use the *instance ID* as both username and database name (see your Aura `.txt` export).

In [ ]:
def get_neo4j_creds():
    keys = ["NEO4J_URI","NEO4J_USERNAME","NEO4J_PASSWORD","NEO4J_DATABASE"]
    try:                                    # Colab Secrets
        from google.colab import userdata
        for k in keys:
            try: os.environ.setdefault(k, userdata.get(k))
            except Exception: pass
    except ImportError: pass
    if not all(os.environ.get(k) for k in keys[:3]): return None
    return {k:os.environ.get(k) for k in keys} | {"NEO4J_DATABASE": os.environ.get("NEO4J_DATABASE") or os.environ["NEO4J_USERNAME"]}

def sync_to_neo4j(frame, creds, batch=2000):
    from neo4j import GraphDatabase
    q = """UNWIND $rows AS r
            MERGE (a:Account {id:r.o}) MERGE (b:Account {id:r.d})
            CREATE (a)-[:TRANSACTED {step:r.step, amount:r.amount, type:r.type, score:r.score, fraud:r.fraud}]->(b)"""
    rows = [dict(o=o,d=d,step=int(s),amount=float(a),type=t,score=float(sc),fraud=int(f)) for o,d,s,a,t,sc,f in
            zip(frame.nameOrig,frame.nameDest,frame.step,frame.amount,frame.type,frame.score,frame.isFraud)]
    driver = GraphDatabase.driver(creds["NEO4J_URI"], auth=(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"]))
    try:
        driver.verify_connectivity()
        driver.execute_query("CREATE CONSTRAINT acct IF NOT EXISTS FOR (a:Account) REQUIRE a.id IS UNIQUE", database_=creds["NEO4J_DATABASE"])
        for i in range(0, len(rows), batch):
            driver.execute_query(q, rows=rows[i:i+batch], database_=creds["NEO4J_DATABASE"])
        print(f"synced {len(rows)} transactions")
    finally:
        driver.close()                      # always release the connection

creds = get_neo4j_creds()
if creds is None: print("Neo4j creds not set -> skipping (set NEO4J_URI / NEO4J_USERNAME / NEO4J_PASSWORD).")
else:
    try: sync_to_neo4j(te[te.flag==1], creds)      # push only flagged test transactions (keeps free tier small)
    except Exception as e: print("Neo4j sync failed:", type(e).__name__, str(e)[:300])

## 7. Reading the results honestly

- Compare **PR-AUC** (not accuracy) of the hybrid vs. the gradient-boosting baseline on the held-out *future* period.
- On real PaySim, expect the tabular baseline to be very strong (balance-error leakage). If the graph model does not beat it, say so — that's a legitimate finding, and the pipeline is still the deliverable.
- PaySim has few repeated counterparties, so Louvain rings will be sparse on real data. Use the degree check above before claiming "syndicates".
- The paper's custom metrics are **not** reproduced here; `f1` is plain F1 and the cycle count is a count, not a recall.